In [3]:
import sys
!{sys.executable} -m pip uninstall -y typing_extensions
!{sys.executable} -m pip install --no-cache-dir typing_extensions

Found existing installation: typing_extensions 4.15.0
Uninstalling typing_extensions-4.15.0:
  Successfully uninstalled typing_extensions-4.15.0

[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip


In [1]:
!pip install typing spacy matplotlib textdescriptives ripser transformers accelerate torch numpy pandas tqdm networkx hf_transfer datasets seaborn pyarrow fastparquet datasets
!python -m spacy download en_core_web_sm
!pip install dcor


[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')

[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
from huggingface_hub import login
login()

In [4]:
import torch
from transformers import (
    BertTokenizer, 
    BertModel,
    GPT2Tokenizer,
    GPT2Model,
    AutoTokenizer,
    AutoModelForCausalLM
)
import numpy as np
import pandas as pd
import ripser
from ripser import ripser
from tqdm import tqdm
import math
import numpy as geek
import networkx as nx
from datasets import load_dataset
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Optional, List, Tuple
import spacy
import textdescriptives as td
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import dcor

In [5]:
# ============================================================================
# TDA COMPUTATION FUNCTIONS
# ============================================================================

def find_highest_finite_value_comprehension(data):
    """Finds the highest value in a list, ignoring inf values, using list comprehension."""
    finite_values = [x for x in data if not math.isinf(x)]
    return max(finite_values) if finite_values else -math.inf

def get_second_value_ignoring_inf(data):
    """
    Returns the second non-inf value in a list.

    Args:
      data: A list of numerical values.

    Returns:
      The second non-inf value in the list, or None if not found.
    """
    non_inf_values = [x for x in data if not math.isinf(x)]
    if len(non_inf_values) < 2:
        return None
    return non_inf_values[1]

def get_bert_attention(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # Move to GPU
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
    attention_matrices = torch.stack(outputs.attentions).mean(dim=0).squeeze()
    attention_matrices = attention_matrices.float().cpu()  # Move back to CPU for numpy
    return np.mean(attention_matrices.numpy(), axis=0)

def get_qwen_attention(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # Move to GPU
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
    attention_matrices = torch.stack(outputs.attentions).mean(dim=0).squeeze()
    attention_matrices = attention_matrices.float().cpu()  # Move back to CPU
    return np.mean(attention_matrices.numpy(), axis=0)

def get_llama_attention(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # Move to GPU
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
    attention_matrices = torch.stack(outputs.attentions).mean(dim=0).squeeze()
    attention_matrices = attention_matrices.float().cpu()  # Move back to CPU
    return np.mean(attention_matrices.numpy(), axis=0)

def get_gpt_attention(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # Move to GPU
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
    attention_matrices = torch.stack(outputs.attentions).mean(dim=0).squeeze()
    attention_matrices = attention_matrices.float().cpu()  # Move back to CPU
    return np.mean(attention_matrices.numpy(), axis=0)
    
def build_graph(attention_matrix, threshold=0.1, causal=False):
    if causal:
        graph = nx.DiGraph()  # Directed graph for causal attention
        num_nodes = attention_matrix.shape[0]
        
        for i in range(num_nodes):
            for j in range(num_nodes):
                if attention_matrix[i, j] > threshold:
                    graph.add_edge(i, j, weight=attention_matrix[i, j])
    else:
        graph = nx.Graph()  # Undirected for bidirectional
        num_nodes = attention_matrix.shape[0]
        
        for i in range(num_nodes):
            for j in range(i + 1, num_nodes):
                if attention_matrix[i, j] > threshold:
                    graph.add_edge(i, j, weight=attention_matrix[i, j])
    
    return graph

def compute_tda_features(graph):
    adjacency_matrix = nx.to_scipy_sparse_array(graph, format='csr')
    diagrams = ripser(adjacency_matrix, maxdim=1)['dgms']
    
    h0 = diagrams[0]
    h1 = diagrams[1] if len(diagrams) > 1 else np.array([])
    
    num_h0 = np.count_nonzero(np.round(h0))
    highest_h0 = find_highest_finite_value_comprehension(h0[:, 1] - h0[:, 0]) if num_h0 > 0 else 0
    Second_highest_h0 = get_second_value_ignoring_inf(h0[:, 1] - h0[:, 0]) if num_h0 > 1 else 0
    highest_minus_second_h0 = highest_h0 - Second_highest_h0 if num_h0 > 1 else 0
    
    h0[np.isinf(h0)] = 0
    mean_h0 = np.mean(h0) if num_h0 > 0 else 0
    
    num_h1 = np.count_nonzero(np.round(h1))
    highest_h1 = find_highest_finite_value_comprehension(h1[:, 1] - h1[:, 0]) if num_h1 > 0 else 0
    second_highest_h1 = get_second_value_ignoring_inf(h1[:, 1] - h1[:, 0]) if num_h1 > 1 else 0
    highest_minus_second_h1 = highest_h1 - second_highest_h1 if num_h1 > 1 else 0
    
    h1[np.isinf(h1)] = 0
    mean_h1 = np.mean(h1) if num_h1 > 0 else 0
    
    h0_persistences = np.sort(h0[:, 1] - h0[:, 0]) if num_h0 > 0 else np.array([])
    
    h1_persistences = np.sort(h1[:, 1] - h1[:, 0]) if num_h1 > 0 else np.array([])
    
    sum_persistence_0 = np.sum(h0_persistences) if len(h0_persistences) > 0 else 0
    sum_persistence_1 = np.sum(h1_persistences) if len(h1_persistences) > 0 else 0
    persistence_entropy_0 = -np.sum(h0_persistences * np.log(h0_persistences + 1e-10)) if len(h0_persistences) > 0 else 0
    persistence_entropy_1 = -np.sum(h1_persistences * np.log(h1_persistences + 1e-10)) if len(h1_persistences) > 0 else 0
    
    betti_curve_0 = len(h0_persistences) if num_h0 > 0 else 0
    betti_curve_1 = len(h1_persistences) if num_h1 > 0 else 0
    
    return [num_h0, highest_h0, highest_minus_second_h0, mean_h0, betti_curve_0, persistence_entropy_0,
            num_h1, highest_h1, highest_minus_second_h1, mean_h1, betti_curve_1, persistence_entropy_1]

def process_texts(texts, model_type='bert', model_path='bert-base-uncased'):
    import warnings
    
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*unexpected.*", category=UserWarning)
        
        if model_type == 'bert':
            tokenizer = BertTokenizer.from_pretrained(model_path)
            model = BertModel.from_pretrained(model_path, attn_implementation="eager")
            model = model.to('cuda') 
            attention_func = get_bert_attention
            causal = False
        elif model_type == 'gpt':
            tokenizer = GPT2Tokenizer.from_pretrained(model_path)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
                tokenizer.pad_token_id = tokenizer.eos_token_id  # Add this
            model = GPT2Model.from_pretrained(model_path, attn_implementation="eager")
            model.config.pad_token_id = tokenizer.pad_token_id  # Add this
            model = model.to('cuda') 
            attention_func = get_gpt_attention
            causal = True
        elif model_type == 'llama':
            tokenizer = AutoTokenizer.from_pretrained(model_path)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
                tokenizer.pad_token_id = tokenizer.eos_token_id  # Add this
            model = AutoModelForCausalLM.from_pretrained(model_path, attn_implementation="eager")
            model.config.pad_token_id = tokenizer.pad_token_id  # Add this
            model = model.to('cuda') 
            attention_func = get_llama_attention
            causal = True
        elif model_type == 'qwen':
            tokenizer = AutoTokenizer.from_pretrained(model_path)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
                tokenizer.pad_token_id = tokenizer.eos_token_id  # Add this
            model = AutoModelForCausalLM.from_pretrained(model_path, attn_implementation="eager")
            model.config.pad_token_id = tokenizer.pad_token_id  # Add this
            model = model.to('cuda') 
            attention_func = get_qwen_attention
            causal = True
    
    data = []
    for text in tqdm(texts):
        attention_matrix = attention_func(text, model, tokenizer)
        graph = build_graph(attention_matrix, causal=causal)
        tda_features = compute_tda_features(graph)
        data.append(tda_features)
    
    columns = ["Num_0dim", "Max_0dim", "Max_0dim_Minus_Second", "Mean_0dim", 
               "betti_curve_0", "persistence_entropy_0",
               "Num_1dim", "Max_1dim", "Max_1dim_Minus_Second", "Mean_1dim", 
               "betti_curve_1", "persistence_entropy_1"]
    return pd.DataFrame(data, columns=columns)

texts = ["This is a test sentence.", "Another example of text processing."]
df = process_texts(texts, "bert")
print(df)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/ripser/ripser.py:251: UserWarning: The input matrix is square, but the distance_matrix flag is off.  Did you mean to indicate that this was a distance matrix?
  warnings.warn(
100%|██████████| 2/2 [

   Num_0dim  Max_0dim  Max_0dim_Minus_Second  Mean_0dim  betti_curve_0  \
0         2  0.763828               0.684179   0.099308              8   
1         2  1.240708               1.123605   0.120323              8   

   persistence_entropy_0  Num_1dim  Max_1dim  Max_1dim_Minus_Second  \
0               1.690237         0         0                      0   
1               1.194267         0         0                      0   

   Mean_1dim  betti_curve_1  persistence_entropy_1  
0          0              0                      0  
1          0              0                      0  


In [6]:
# ============================================================================
# LINGUISTIC FEATURE EXTRACTION
# ============================================================================

def extract_linguistic_features(texts: pd.Series, nlp_model: str = "en_core_web_sm") -> pd.DataFrame:
    nlp = spacy.load(nlp_model)
    nlp.add_pipe("textdescriptives/all")
    
    results = []
    for text in tqdm(texts, desc="Extracting linguistic features"):
        try:
            doc = nlp(text)
            features = td.extract_df(doc)
            results.append(features.iloc[0])
        except Exception as e:
            print(f"Error processing text: {e}")
            # Add empty row with NaN values
            results.append(pd.Series())
    
    return pd.concat(results, axis=1).T.reset_index(drop=True)

In [7]:
# ============================================================================
# CORRELATION ANALYSIS
# ============================================================================

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()
    df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
    df_clean = df_clean.fillna(df_clean.mean())
    df_clean = df_clean.fillna(0)
    df_clean = df_clean.astype(float)
    return df_clean

def compute_dcor_matrix(tda_data, linguistic_data):
    tda_clean = clean_data(tda_data)
    ling_clean = clean_data(linguistic_data)
    
    tda_features = tda_clean.to_numpy()
    linguistic_features = ling_clean.to_numpy()
    
    corr_matrix = np.zeros((tda_features.shape[1], linguistic_features.shape[1]))
    for i in range(tda_features.shape[1]):
        for j in range(linguistic_features.shape[1]):
            try:
                corr_matrix[i, j] = dcor.distance_correlation(tda_features[:, i], linguistic_features[:, j])
            except Exception as e:
                print(f"Warning: Could not compute correlation for indices ({i}, {j}): {e}")
                corr_matrix[i, j] = 0.0
    
    return corr_matrix
    
def plot_category_heatmap(corr_matrix: np.ndarray, 
                          tda_feature_names: List[str],
                          linguistic_feature_names: List[str], 
                          title: str, 
                          figsize: Tuple[int, int] = (8, 4)) -> plt.Figure:
    """Plot heatmap for a specific category"""
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt='.3f',
                xticklabels=linguistic_feature_names,
                yticklabels=tda_feature_names,
                vmin=0, vmax=1, ax=ax,
                annot_kws={'size': 8})
    ax.set_title(title, fontsize=11)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    return fig

In [8]:
FEATURE_CATEGORIES = {
    'descriptive_statistics': [
        'token_length_mean', 'token_length_median', 'token_length_std',
        'sentence_length_mean', 'sentence_length_median', 'sentence_length_std',
        'syllables_per_token_mean', 'syllables_per_token_median', 'syllables_per_token_std',
        'n_tokens', 'n_unique_tokens', 'proportion_unique_tokens', 
        'n_characters', 'n_sentences'
    ],
    'readability': [
        'flesch_reading_ease', 'flesch_kincaid_grade', 
        'smog', 'gunning_fog', 'automated_readability_index',
        'coleman_liau_index', 'lix', 'rix'
    ],
    'pos_proportions': [
        'pos_prop_NOUN', 'pos_prop_VERB', 'pos_prop_ADJ', 'pos_prop_ADV',
        'pos_prop_PRON', 'pos_prop_DET', 'pos_prop_ADP', 'pos_prop_AUX',
        'pos_prop_CONJ', 'pos_prop_PROPN'
    ],
    'dependency': [
        'dependency_distance_mean', 'dependency_distance_std',
        'prop_adjacent_dependency_relation_mean', 'prop_adjacent_dependency_relation_std'
    ],
    'information_theory': [
        'entropy', 'per_word_entropy', 'perplexity'
    ],
    'coherence': [
        'coherence_mean', 'coherence_std'
    ]
}

def analyze_dataset(dataset_name: str,
                   dataset_config: str,
                   text_column: str,
                   label_column: str,
                   model_type: str = "llama",
                   model_path: str = "meta-llama/Llama-3.1-8B",
                   n_samples: int = 500,
                   output_pdf: str = "TDA_Linguistic_Heatmaps.pdf",
                   tda_features: Optional[List[str]] = None):
    
    if tda_features is None:
        tda_features = ["Num_0dim", "Max_0dim", "Mean_0dim", 
                       "betti_curve_0", "persistence_entropy_0"]
    
    print("="*60)
    print(f"TDA-Linguistic Analysis Pipeline")
    print(f"Dataset: {dataset_name}/{dataset_config}")
    print(f"Model: {model_type} - {model_path}")
    print("="*60)
    
    print("\n1. Loading dataset...")
    ds = load_dataset(dataset_name, dataset_config, split="train")
    df = ds.to_pandas()
    
    unique_labels = df[label_column].unique()
    print(f"Found {len(unique_labels)} unique labels: {unique_labels}")
    
    results = {}
    for label in unique_labels:
        print(f"\n{'='*60}")
        print(f"Processing label: {label}")
        print(f"{'='*60}")
        
        texts = df[df[label_column] == label][text_column].head(n_samples)
        print(f"Number of texts: {len(texts)}")
        
        print("\n2. Extracting TDA features...")
        tda_df = process_texts(texts.tolist(), model_type=model_type, model_path=model_path)
        
        print("\n3. Extracting linguistic features...")
        linguistic_df = extract_linguistic_features(texts)
        
        results[label] = {
            'tda': tda_df,
            'linguistic': linguistic_df
        }
    
    print(f"\n{'='*60}")
    print("4. Computing correlations and generating heatmaps...")
    print(f"{'='*60}")
    
    with PdfPages(output_pdf) as pdf:
        for category, features in FEATURE_CATEGORIES.items():
            print(f"\nProcessing category: {category}")
            
            sample_ling = list(results.values())[0]['linguistic']
            available_features = [f for f in features if f in sample_ling.columns]
            
            if len(available_features) == 0:
                print(f"No features available for {category}, skipping...")
                continue
            
            for label in unique_labels:
                tda_data = results[label]['tda'][tda_features]
                ling_data = results[label]['linguistic'][available_features]
                
                # CONVERT BOOLEAN COLUMNS TO NUMERIC
                ling_data = ling_data.apply(pd.to_numeric, errors='coerce')
                
                # DROP COLUMNS THAT COULDN'T BE CONVERTED
                ling_data = ling_data.dropna(axis=1, how='all')
                
                if ling_data.empty:
                    print(f"No numeric features for {category} label {label}, skipping...")
                    continue
                
                corr_matrix = compute_dcor_matrix(tda_data, ling_data)
                
                fig = plot_category_heatmap(
                    corr_matrix,
                    tda_features,
                    ling_data.columns.tolist(),  # Use actual columns after conversion
                    f"Label {label} - {category.replace('_', ' ').title()}"
                )
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
            
            if len(unique_labels) == 2:
                labels = list(unique_labels)
                
                # Convert for both labels
                ling_data_0 = results[labels[0]]['linguistic'][available_features].apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all')
                ling_data_1 = results[labels[1]]['linguistic'][available_features].apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all')
                
                # Only compute difference if both have data
                if not ling_data_0.empty and not ling_data_1.empty:
                    # Use intersection of columns
                    common_cols = list(set(ling_data_0.columns) & set(ling_data_1.columns))
                    
                    if common_cols:
                        corr_0 = compute_dcor_matrix(
                            results[labels[0]]['tda'][tda_features],
                            ling_data_0[common_cols]
                        )
                        corr_1 = compute_dcor_matrix(
                            results[labels[1]]['tda'][tda_features],
                            ling_data_1[common_cols]
                        )
                        
                        corr_diff = corr_0 - corr_1
                        
                        fig, ax = plt.subplots(figsize=(8, 4))  # Changed from (12, 6)
                        sns.heatmap(corr_diff, annot=True, cmap="RdBu_r", fmt='.3f', 
                                   center=0, xticklabels=common_cols,
                                   yticklabels=tda_features, vmin=-1, vmax=1, ax=ax,
                                   annot_kws={'size': 8})  # Added
                        ax.set_title(f"Difference: Label {labels[0]} - Label {labels[1]} - "
                                   f"{category.replace('_', ' ').title()}", 
                                   fontsize=11)  # Changed from 14, removed fontweight='bold'
                        plt.xticks(rotation=45, ha='right', fontsize=8)  # Added fontsize
                        plt.yticks(fontsize=8)  # Added fontsize
                        plt.tight_layout()
                        pdf.savefig(fig, bbox_inches='tight')
                        plt.close(fig)
    
    print(f"\n{'='*60}")
    print("ANALYSIS COMPLETE!")
    print(f"Heatmaps saved to: {output_pdf}")
    print(f"{'='*60}")
    
    return results

In [9]:
"""
Run TDA-Linguistic analysis across all dataset-model combinations
"""

if __name__ == "__main__":
    
    # Define all models with their types
    models = {
        "qwen-2.5-7b-instruct": {"type": "qwen", "path": "Qwen/Qwen2.5-7B-Instruct"}
    }
    
    models = {
        "llama-3.1-8b-instruct": {"type": "llama", "path": "meta-llama/Llama-3.1-8B-Instruct"},
        "qwen-2.5-7b-instruct": {"type": "qwen", "path": "Qwen/Qwen2.5-7B-Instruct"},
        "gpt2": {"type": "gpt", "path": "gpt2"},
        "distilgpt2": {"type": "gpt", "path": "distilgpt2"},
        "roberta-base": {"type": "bert", "path": "roberta-base"},
        "bert-base-uncased": {"type": "bert", "path": "bert-base-uncased"},
        "distilbert-base-uncased": {"type": "bert", "path": "distilbert-base-uncased"},
        "electra-small": {"type": "bert", "path": "google/electra-small-discriminator"}
    }

        # {"name": "nyu-mll/glue", "config": "cola", "text_column": "sentence", 
        #  "label_column": "label", "description": "GLUE-CoLA"},
        # {"name": "nyu-mll/glue", "config": "sst2", "text_column": "sentence", 
        #  "label_column": "label", "description": "GLUE-SST2"},
        # {"name": "nyu-mll/glue", "config": "mrpc", "text_column": "sentence1", 
        #  "label_column": "label", "description": "GLUE-MRPC"},
        # {"name": "nyu-mll/glue", "config": "qqp", "text_column": "question1", 
        #  "label_column": "label", "description": "GLUE-QQP"},
        # {"name": "nyu-mll/glue", "config": "mnli", "text_column": "premise", 
        #  "label_column": "label", "description": "GLUE-MNLI"},
        # {"name": "nyu-mll/glue", "config": "qnli", "text_column": "question", 
        #  "label_column": "label", "description": "GLUE-QNLI"},
        # {"name": "nyu-mll/glue", "config": "rte", "text_column": "sentence1", 
        #  "label_column": "label", "description": "GLUE-RTE"},
        # {"name": "nyu-mll/glue", "config": "wnli", "text_column": "sentence1", 
        #  "label_column": "label", "description": "GLUE-WNLI"},
        # {"name": "super_glue", "config": "boolq", "text_column": "question", 
        #  "label_column": "label", "description": "SuperGLUE-BoolQ"},
        # {"name": "super_glue", "config": "cb", "text_column": "premise", 
        #  "label_column": "label", "description": "SuperGLUE-CB"},
        # {"name": "super_glue", "config": "copa", "text_column": "premise", 
        #  "label_column": "label", "description": "SuperGLUE-COPA"},
        # {"name": "super_glue", "config": "wic", "text_column": "sentence1", 
        #  "label_column": "label", "description": "SuperGLUE-WiC"},
        # {"name": "super_glue", "config": "wsc", "text_column": "text", 
        #  "label_column": "label", "description": "SuperGLUE-WSC"},
    
    datasets = [
        {"name": "snli", "config": "plain_text", "text_column": "premise",  # Changed from None
         "label_column": "label", "description": "SNLI"},
        {"name": "multi_nli", "config": "plain_text", "text_column": "premise",  # Changed from None
         "label_column": "label", "description": "MultiNLI"},
        {"name": "blimp", "config": "anaphor_agreement", "text_column": "sentence_good", 
         "label_column": None, "description": "BLiMP-Anaphor"},
        {"name": "blimp", "config": "argument_structure", "text_column": "sentence_good", 
         "label_column": None, "description": "BLiMP-Argument"},
        {"name": "winogrande", "config": "winogrande_xl", "text_column": "sentence", 
         "label_column": "answer", "description": "Winogrande-XL"},
    ]
    
    import os
    from datetime import datetime
    
    results_dir = "tda_linguistic_results"
    os.makedirs(results_dir, exist_ok=True)
    
    log_file = os.path.join(results_dir, f"run_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
    
    total_combinations = len(datasets) * len(models)
    current = 0
    
    with open(log_file, 'w') as log:
        log.write(f"TDA-Linguistic Analysis | Total: {total_combinations} | "
                 f"Datasets: {len(datasets)} | Models: {len(models)}\n{'='*80}\n\n")
        
        for dataset_config in datasets:
            for model_name, model_info in models.items():
                current += 1
                
                if dataset_config["label_column"] is None:
                    log.write(f"[{current}/{total_combinations}] SKIPPED: "
                             f"{dataset_config['description']} - {model_name} (no labels)\n")
                    continue
                
                output_filename = f"{dataset_config['description']}_{model_name}_TDA_Linguistic.pdf"
                output_path = os.path.join(results_dir, output_filename)
                
                print(f"[{current}/{total_combinations}] {dataset_config['description']} - {model_name}")
                log.write(f"[{current}/{total_combinations}] STARTING: "
                         f"{dataset_config['description']} - {model_name}\n")
                log.flush()
                
                try:
                    config = dataset_config["config"] if dataset_config["config"] is not None else "default"
                    
                    results = analyze_dataset(
                        dataset_name=dataset_config["name"],
                        dataset_config=config,
                        text_column=dataset_config["text_column"],
                        label_column=dataset_config["label_column"],
                        model_type=model_info["type"],
                        model_path=model_info["path"],
                        n_samples=50,
                        output_pdf=output_path
                    )
                    
                    log.write(f"[{current}/{total_combinations}] SUCCESS: "
                             f"{dataset_config['description']} - {model_name}\n")
                    
                except Exception as e:
                    log.write(f"[{current}/{total_combinations}] FAILED: "
                             f"{dataset_config['description']} - {model_name} - {str(e)}\n")
                    print(f"  ✗ Error: {str(e)}")
                    continue
                
                log.flush()
    
    print(f"\nComplete! Results: {results_dir}/ | Log: {log_file}")

[1/40] SNLI - llama-3.1-8b-instruct
TDA-Linguistic Analysis Pipeline
Dataset: snli/plain_text
Model: llama - meta-llama/Llama-3.1-8B-Instruct

1. Loading dataset...


plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/412k [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/413k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

...

Loading weights: 0it [00:00, ?it/s]

BertModel LOAD REPORT from: distilbert-base-uncased
Key                                                                      | Status     | 
-------------------------------------------------------------------------+------------+-
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.ffn.lin2.weight          | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.sa_layer_norm.bias       | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.k_lin.bias     | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.out_lin.weight | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.v_lin.bias     | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.q_lin.bias     | UNEXPECTED | 
vocab_transform.weight                                                   | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.output_layer_norm.bias   | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.ffn.lin2.b

  ✗ Error: Graph has no nodes or edges
[40/40] Winogrande-XL - electra-small
TDA-Linguistic Analysis Pipeline
Dataset: winogrande/winogrande_xl
Model: bert - google/electra-small-discriminator

1. Loading dataset...
Found 2 unique labels: <ArrowStringArray>
['2', '1']
Length: 2, dtype: str

Processing label: 2
Number of texts: 50

2. Extracting TDA features...


You are using a model of type electra to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.


  ✗ Error: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

Complete! Results: tda_linguistic_results/ | Log: tda_linguistic_results/run_log_20260204_222004.txt
